# RAG Avancé – RGPD
## Architecture : Multi-Query → Parent Document Retriever → Re-ranker → LLM

```
Question utilisateur
        │
Multi-Query Retriever  (reformule en N variantes)
        │
Vector Store (chunks)  +  métadonnées (chapitre, titre, article)
        │
Parent Document Retriever  (renvoie l'article complet au LLM)
        │
Re-ranker  (cross-encoder pour trier les articles)
        │
LLM HuggingFace (Qwen2.5-0.5B-Instruct)
        │
Réponse structurée + articles cités
```


## 1. Imports & dépendances

```bash
pip install langchain langchain-community langchain-chroma chromadb sentence-transformers transformers accelerate beautifulsoup4 requests
```

In [1]:
import requests
import json
import re
import uuid

from bs4 import BeautifulSoup
from dotenv import load_dotenv

load_dotenv()


True

## 2. Chargement et parsing du RGPD

On extrait chaque article avec ses métadonnées : numéro de chapitre, titre du chapitre, numéro d'article, titre d'article.

In [2]:
with open("L_2016119FR.01000101.html", "r", encoding="utf-8") as f:
    html = f.read()

soup = BeautifulSoup(html, "html.parser")


def clean_text(element):
    if element is None:
        return ""
    return element.get_text(" ", strip=True)


resultat = []

chapitres = soup.find_all("div", id=re.compile(r"^cpt_[IVXLCDM]+$"))

for chapitre in chapitres:
    chapitre_nom   = clean_text(chapitre.find("p", class_="oj-ti-section-1"))
    titre_chapitre = clean_text(chapitre.find("p", class_="oj-ti-section-2"))

    if not chapitre_nom.startswith("CHAPITRE"):
        continue

    chapitre_data = {
        "chapitre":       chapitre_nom,
        "titre_chapitre": titre_chapitre,
        "contenu":        []
    }

    articles         = chapitre.find_all("div", id=re.compile(r"^art_\d+$"))
    articles_deja_vus = set()

    for article in articles:
        article_id = article.get("id")
        if article_id in articles_deja_vus:
            continue
        articles_deja_vus.add(article_id)

        numero_article = clean_text(article.find("p", class_="oj-ti-art"))
        titre_article  = clean_text(article.find("p", class_="oj-sti-art"))

        if not numero_article.startswith("Article"):
            continue

        paragraphes     = article.find_all("p", class_="oj-normal")
        contenu_article = "\n".join(
            clean_text(p) for p in paragraphes if clean_text(p)
        )

        chapitre_data["contenu"].append({
            "article":         numero_article,
            "titre_article":   titre_article,
            "contenu_article": contenu_article
        })

    resultat.append(chapitre_data)

print(f"✅  {len(resultat)} chapitres extraits")
print(f"✅  {sum(len(c['contenu']) for c in resultat)} articles extraits")

with open("rgpd_structure.json", "w", encoding="utf-8") as f:
    json.dump(resultat, f, ensure_ascii=False, indent=4)


✅  11 chapitres extraits
✅  99 articles extraits


## 3. Construction des Documents LangChain avec métadonnées

### Stratégie Parent / Child
- **Child chunks** (small) : utilisés pour la recherche vectorielle (chunk_size=300, overlap=50)
- **Parent documents** : l'article complet, renvoyé au LLM pour la génération

Chaque chunk enfant porte les métadonnées :
```python
{
  "chapitre":         "CHAPITRE II",
  "titre_chapitre":   "Principes",
  "article":          "Article 5",
  "titre_article":    "Principes relatifs au traitement des données",
  "type":             "child_chunk",
  "parent_id":        "<uuid>",   # lien vers l'article complet
  "source":           "RGPD"
}
```




In [3]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

with open("rgpd_structure.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# ── Splitter pour les child chunks ──────────────────────────────────
# chunk_size=300 : assez grand pour capturer le sens d'un paragraphe RGPD
# chunk_overlap=50 : évite de couper une phrase entre deux chunks
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

parent_documents = []   # articles complets => envoyés au LLM
child_documents  = []   # petits chunks => indexés dans le vector store

for chapitre in data:
    chapitre_nom   = chapitre.get("chapitre", "")
    titre_chapitre = chapitre.get("titre_chapitre", "")

    for article in chapitre.get("contenu", []):
        numero_article  = article.get("article", "")
        titre_article   = article.get("titre_article", "")
        contenu_article = article.get("contenu_article", "")

        if not contenu_article.strip():
            continue

        # Texte complet de l'article (parent)
        parent_text = (
            f"{chapitre_nom} – {titre_chapitre}\n"
            f"{numero_article} – {titre_article}\n\n"
            f"{contenu_article}"
        )
        parent_id = str(uuid.uuid4())

        # Métadonnées partagées
        metadata_base = {
            "chapitre":         chapitre_nom,
            "titre_chapitre":   titre_chapitre,
            "article":          numero_article,
            "titre_article":    titre_article,
            "source":           "RGPD",
            "parent_id":        parent_id,
        }

        # Document parent (article complet)
        parent_doc = Document(
            page_content=parent_text,
            metadata={**metadata_base, "type": "parent"}
        )
        parent_documents.append(parent_doc)

        # Child chunks (pour la recherche)
        chunks = child_splitter.split_text(contenu_article)
        for i, chunk in enumerate(chunks):
            child_doc = Document(
                page_content=chunk,
                metadata={**metadata_base, "type": "child_chunk", "chunk_index": i}
            )
            child_documents.append(child_doc)

print(f"✅  {len(parent_documents)} articles parents")
print(f"✅  {len(child_documents)} child chunks créés")
print()
print("Exemple de métadonnées d'un child chunk :")
print(json.dumps(child_documents[0].metadata, ensure_ascii=False, indent=2))


c:\Users\berul\OneDrive\Bureau\AI31 Projet\AI31_Projet_RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅  99 articles parents
✅  1067 child chunks créés

Exemple de métadonnées d'un child chunk :
{
  "chapitre": "CHAPITRE I",
  "titre_chapitre": "Dispositions générales",
  "article": "Article premier",
  "titre_article": "Objet et objectifs",
  "source": "RGPD",
  "parent_id": "77283f95-d61e-405f-9618-a116dfcecf4a",
  "type": "child_chunk",
  "chunk_index": 0
}


## Comprendre la stratégie Parent / Child

### Document Parent
```python
Document(
    metadata={
        'chapitre':       'CHAPITRE I',
        'titre_chapitre': 'Dispositions générales',
        'article':        'Article premier',
        'titre_article':  'Objet et objectifs',
        'type':           'parent',          # ← identifiant
        'parent_id':      '693b872e-...',
        'source':         'RGPD'
    },
    page_content="CHAPITRE I – Dispositions générales\nArticle premier – ...\n\n[article complet]"
)
```
**C'est l'article RGPD en entier.**
Il est envoyé tel quel au LLM pour la génération de la réponse.
Le LLM a ainsi tout le contexte juridique sans perte d'information.

---

### Document Child (chunk)
```python
Document(
    metadata={
        'chapitre':       'CHAPITRE I',
        'titre_chapitre': 'Dispositions générales',
        'article':        'Article premier',
        'titre_article':  'Objet et objectifs',
        'type':           'child_chunk',     # ← identifiant
        'parent_id':      '693b872e-...',    # ← lien vers le parent
        'chunk_index':    0,                 # ← position dans l'article
        'source':         'RGPD'
    },
    page_content="1. Le présent règlement établit des règles relatives à la protection..."
)
```
**C'est un petit morceau de l'article** (≈300 tokens).
Il est indexé dans le **Vector Store** et utilisé pour la recherche sémantique.
Le `parent_id` permet de remonter immédiatement à l'article complet une fois trouvé.

---

### Pourquoi cette séparation ?

| | Child chunk | Parent document |
|---|---|---|
| **Rôle** | Recherche sémantique | Génération LLM |
| **Taille** | ~300 tokens | Article entier |
| **Stocké dans** | Vector Store (ChromaDB) | Dictionnaire `parent_index` |
| **Précision** | Fine (petit = précis) |  Complet (pas de perte) |



In [4]:
child_documents

[Document(metadata={'chapitre': 'CHAPITRE I', 'titre_chapitre': 'Dispositions générales', 'article': 'Article premier', 'titre_article': 'Objet et objectifs', 'source': 'RGPD', 'parent_id': '77283f95-d61e-405f-9618-a116dfcecf4a', 'type': 'child_chunk', 'chunk_index': 0}, page_content="1.\xa0\xa0\xa0Le présent règlement établit des règles relatives à la protection des personnes physiques à l'égard du traitement des données à caractère personnel et des règles relatives à la libre circulation de ces données."),
 Document(metadata={'chapitre': 'CHAPITRE I', 'titre_chapitre': 'Dispositions générales', 'article': 'Article premier', 'titre_article': 'Objet et objectifs', 'source': 'RGPD', 'parent_id': '77283f95-d61e-405f-9618-a116dfcecf4a', 'type': 'child_chunk', 'chunk_index': 1}, page_content='2.\xa0\xa0\xa0Le présent règlement protège les libertés et droits fondamentaux des personnes physiques, et en particulier leur droit à la protection des données à caractère personnel.'),
 Document(met

In [5]:
parent_documents

[Document(metadata={'chapitre': 'CHAPITRE I', 'titre_chapitre': 'Dispositions générales', 'article': 'Article premier', 'titre_article': 'Objet et objectifs', 'source': 'RGPD', 'parent_id': '77283f95-d61e-405f-9618-a116dfcecf4a', 'type': 'parent'}, page_content="CHAPITRE I – Dispositions générales\nArticle premier – Objet et objectifs\n\n1.\xa0\xa0\xa0Le présent règlement établit des règles relatives à la protection des personnes physiques à l'égard du traitement des données à caractère personnel et des règles relatives à la libre circulation de ces données.\n2.\xa0\xa0\xa0Le présent règlement protège les libertés et droits fondamentaux des personnes physiques, et en particulier leur droit à la protection des données à caractère personnel.\n3.\xa0\xa0\xa0La libre circulation des données à caractère personnel au sein de l'Union n'est ni limitée ni interdite pour des motifs liés à la protection des personnes physiques à l'égard du traitement des données à caractère personnel."),
 Documen

## 4. Embeddings & Vector Store

On indexe uniquement les **child chunks** dans ChromaDB.
Le modèle `paraphrase-multilingual-MiniLM-L12-v2` est choisi car il supporte nativement le français (bien meilleur que `all-MiniLM-L6-v2` pour du RGPD).

In [6]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# Modèle multilingue : bien adapté au français juridique
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": "cpu"}
)

# On stocke les child chunks dans Chroma
vector_store = Chroma.from_documents(
    documents=child_documents,
    embedding=embedding_model,
    collection_name="rgpd_chunks"
)

# Index parent_id → parent_document pour récupération rapide
parent_index = {doc.metadata["parent_id"]: doc for doc in parent_documents}

print(f"✅  Vector store créé avec {vector_store._collection.count()} chunks")


C:\Users\berul\AppData\Local\Temp\ipykernel_26424\2607370998.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma
C:\Users\berul\AppData\Local\Temp\ipykernel_26424\2607370998.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20279.08it/s]


✅  Vector store créé avec 1067 chunks


## 5. Multi-Query Retriever

Le LLM reformule la question en **4 variantes** pour couvrir plus d'angles sémantiques.
Exemple :
```
Question : "quelles obligations pour traiter des emails ?"
→ "base légale du traitement de données personnelles"
→ "email données personnelles RGPD obligations"
→ "information des personnes concernées traitement"
→ "minimisation des données collecte email"
```


In [ ]:
from langchain_community.llms import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# ── LLM pour la reformulation Multi-Query ───────────────────────────
# Qwen2.5-0.5B-Instruct : léger, gratuit, instruct-tuned
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model_hf  = AutoModelForCausalLM.from_pretrained(model_name)

pipe = pipeline(
    "text-generation",
    model=model_hf,
    tokenizer=tokenizer,
    max_new_tokens=356,
    temperature=0.3,
    
#     temperature=0.0   →  déterministe, répétitif
# temperature=0.3   →  factuel, stable       ← ton choix ✅
# temperature=0.7   →  créatif, variable
# temperature=1.0   →  aléatoire, hallucinations fréquentes
    do_sample=True,
)

llm = HuggingFacePipeline(pipeline=pipe)


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 4935.94it/s]


In [8]:
from langchain_core.prompts import PromptTemplate

# Prompt de reformulation en français
multi_query_prompt = PromptTemplate(
    input_variables=["question"],
    template="""Tu es un assistant spécialisé en droit RGPD.
Ta tâche est de reformuler la question suivante en 4 variantes différentes,
pour améliorer la recherche dans une base documentaire juridique.
Génère uniquement les 4 questions, une par ligne, sans numérotation.

Question originale : {question}
Variantes :"""
)

# Retriever de base (sur les child chunks)
base_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 6}
)

# --- Fonction pour générer les reformulations ---
def generate_queries(question: str, llm, prompt):
    reformulations = llm.invoke(
        prompt.format(question=question)
    )
    # Nettoyage : une ligne = une requête
    return [q.strip() for q in reformulations.split("\n") if q.strip()]

# --- Fonction Multi‑Query moderne ---
def multi_query_search(question: str, retriever, llm, prompt):
    # 1. Générer 4 variantes
    queries = generate_queries(question, llm, prompt)

    all_docs = []
    for q in queries:
        docs = retriever.invoke(q)
        all_docs.extend(docs)

    # 2. Déduplication par contenu
    unique = {doc.page_content: doc for doc in all_docs}
    return list(unique.values())

print("✅ Multi‑Query moderne prêt")


✅ Multi‑Query moderne prêt


## 6. Parent Document Retriever

Après la recherche sur les child chunks, on remonte les **articles complets** (parents).
- Les child chunks donnent les meilleurs matches sémantiques
- Mais on envoie au LLM l'article entier → pas de perte de contexte juridique


In [18]:
def retrieve_parent_documents(question: str, retriever, parent_idx: dict, llm=None, prompt=None, top_k: int = 5):
    """
    1. Cherche les child chunks pertinents via multi_query_search()
    2. Remonte les articles parents correspondants (dédupliqués)
    3. Retourne au max top_k articles parents uniques
    """

    # --- 1. Multi‑query search sur les child chunks ---
    child_hits = multi_query_search(
        question=question,
        retriever=retriever,
        llm=llm,
        prompt=prompt
    )

    # --- 2. Remonter les documents parents ---
    seen_parent_ids = set()
    parent_docs = []

    for child in child_hits:
        pid = child.metadata.get("parent_id")
        if pid and pid not in seen_parent_ids:
            seen_parent_ids.add(pid)
            parent_doc = parent_idx.get(pid)
            if parent_doc:
                parent_docs.append(parent_doc)

        if len(parent_docs) >= top_k:
            break

    return parent_docs

test_docs = retrieve_parent_documents(
    question="Quelles sont les bases légales du traitement ?",
    retriever=base_retriever,
    parent_idx=parent_index,
    llm=llm,
    prompt=multi_query_prompt
)

print(f"✅  {len(test_docs)} articles parents récupérés")
for d in test_docs:
    meta = d.metadata
    print(f"   • {meta['article']} | {meta['chapitre']} – {meta['titre_chapitre']}")


[transformers] Both `max_new_tokens` (=356) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅  5 articles parents récupérés
   • Article 30 | CHAPITRE IV – Responsable du traitement et sous-traitant
   • Article 38 | CHAPITRE IV – Responsable du traitement et sous-traitant
   • Article 28 | CHAPITRE IV – Responsable du traitement et sous-traitant
   • Article 56 | CHAPITRE VI – Autorités de contrôle indépendantes
   • Article 60 | CHAPITRE VII – Coopération et cohérence


## 7. Re-ranker (Cross-Encoder)

Le re-ranker utilise un **cross-encoder** : il évalue la pertinence de chaque article
par rapport à la question (bien plus précis qu'un cosine similarity).

Modèle : `cross-encoder/ms-marco-MiniLM-L-6-v2` — léger et efficace.


In [19]:
from sentence_transformers import CrossEncoder

# Cross-encoder pour le re-ranking
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    max_length=512
)


def rerank_documents(question: str, docs: list, top_n: int = 3):
    """
    Score chaque article avec le cross-encoder,
    retourne les top_n articles les plus pertinents.
    """
    if not docs:
        return []

    # Paires (question, contenu de l'article)
    pairs = [(question, doc.page_content[:512]) for doc in docs]

    scores = reranker.predict(pairs)

    # Tri décroissant par score
    ranked = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)

    return [doc for _, doc in ranked[:top_n]]


# Test
reranked = rerank_documents(
    "Quelles sont les bases légales du traitement ?",
    test_docs
)

print(f"✅  {len(reranked)} articles après re-ranking")
for d in reranked:
    meta = d.metadata
    print(f"   • {meta['article']} – {meta['titre_article']}")


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4490.32it/s]


✅  3 articles après re-ranking
   • Article 38 – Fonction du délégué à la protection des données
   • Article 28 – Sous-traitant
   • Article 30 – Registre des activités de traitement


## 8. Prompt RGPD & RAG Chain finale

### Pipeline complet
```
question
  → Multi-Query Retriever (child chunks)
  → Parent Document Retriever (articles complets)
  → Re-ranker (top 3 articles)
  → LLM + prompt structuré
  → Réponse avec articles cités
```


In [33]:
import re
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

# ── Prompt structuré RGPD ─────────────────────────────────────────────
RGPD_PROMPT = """Tu es un assistant expert en conformité RGPD.
Tu réponds UNIQUEMENT à partir des articles RGPD fournis dans le contexte.
Si l'information n'est pas présente, dis : "Je ne peux pas répondre avec certitude à partir des extraits RGPD fournis."
RÈGLE ABSOLUE : Si l'article mentionné dans la question n'est PAS présent
dans le contexte fourni, réponds UNIQUEMENT : "L'article demandé ne figure
pas dans les extraits fournis. Je ne peux pas répondre."
Ne génère JAMAIS un article qui n'existe pas dans le contexte.

Contexte RGPD (articles complets) :
{context}

Question : {question}

Réponds en suivant EXACTEMENT cette structure :

## 1. Réponse directe
[Réponse claire et concise à la question]

## 2. Articles RGPD concernés
[Liste des articles et chapitres présents dans le contexte]

## 3. Explication détaillée
[Explique ce que disent les articles pertinents]

⚠️ Avertissement : Cette réponse est informative et ne constitue pas un avis juridique.

Réponse :
"""

prompt_template = ChatPromptTemplate.from_template(RGPD_PROMPT)


def format_parent_docs(docs: list) -> str:
    """Formate les articles complets pour le prompt, avec séparateur clair."""
    formatted = []
    for doc in docs:
        meta = doc.metadata
        header = (
            f"{'='*60}\n"
            f"{meta['chapitre']} – {meta['titre_chapitre']}\n"
            f"{meta['article']} – {meta['titre_article']}\n"
            f"{'='*60}"
        )
        formatted.append(f"{header}\n{doc.page_content}")
    return "\n\n".join(formatted)


# ── Point d'entrée intelligent (détecte article précis) ──────────────
def smart_rag_pipeline(question: str) -> dict:
    """
    Routeur principal :
    - Question avec numéro d'article → recherche directe par métadonnées (fiable à 100%)
    - Question sémantique            → pipeline RAG complet
    """
    match = re.search(r"article\s+(\d+|premier)", question, re.IGNORECASE)

    if match:
        # Normalise le numéro : "5" → "Article 5", "premier" → "Article premier"
        numero     = match.group(1).capitalize()
        article_key = f"Article {numero}"

        print(f"\n🎯 Article précis détecté : {article_key}")
        print("─" * 50)

        found = [
            doc for doc in parent_documents
            if doc.metadata["article"].lower() == article_key.lower()
        ]

        if found:
            doc  = found[0]
            meta = doc.metadata
            print(f"✅ Trouvé directement dans les métadonnées")

            reponse = (
                f"## 1. Réponse directe\n"
                f"L'{article_key} se trouve dans le "
                f"{meta['chapitre']} – {meta['titre_chapitre']}.\n\n"
                f"## 2. Article concerné\n"
                f"{meta['article']} – {meta['titre_article']}\n\n"
                f"## 3. Contenu complet\n"
                f"{doc.page_content}"
            )
            return {
                "question":       question,
                "reponse":        reponse,
                "articles_cites": [
                    f"{meta['article']} ({meta['chapitre']} – {meta['titre_chapitre']})"
                ]
            }
        else:
            print(f"❌ Article introuvable dans le RGPD")
            return {
                "question":       question,
                "reponse":        f"{article_key} n'existe pas dans le RGPD (99 articles au total).",
                "articles_cites": []
            }

    # Pas d'article précis détecté → RAG sémantique complet
    print(f"\n🔍 Question sémantique → pipeline RAG complet")
    return full_rag_pipeline(question)



# smart_rag_pipeline("Dans quel chapitre se trouve l'article 99 ?")
#         │
#         ├── re.search détecte "article 99"
#         │       │
#         │       ├── cherche dans parent_documents par métadonnée
#         │       ├── trouve Article 99 → CHAPITRE XI – Dispositions finales
#         │       └── retourne sans appeler le LLM  ✅ fiable à 100%
#         │
# smart_rag_pipeline("Quelles sont les bases légales du traitement ?")
#         │
#         ├── re.search ne détecte rien
#         └── → full_rag_pipeline()  ✅ pipeline complet


# ── Pipeline RAG complet (questions sémantiques) ──────────────────────
def full_rag_pipeline(question: str) -> dict:
    print(f"\n🔍 Question : {question}")
    print("─" * 50)

    parent_docs = retrieve_parent_documents(
        question=question,
        retriever=base_retriever,
        parent_idx=parent_index,
        llm=llm,
        prompt=multi_query_prompt,
        top_k=8
    )
    print(f"📄 {len(parent_docs)} articles parents récupérés")

    reranked_docs = rerank_documents(question, parent_docs, top_n=3)
    print(f"🏆 Top {len(reranked_docs)} articles après re-ranking :")
    articles_cites = []
    for doc in reranked_docs:
        meta = doc.metadata
        label = f"{meta['article']} ({meta['chapitre']} – {meta['titre_chapitre']})"
        print(f"   • {label}")
        articles_cites.append(label)

    context      = format_parent_docs(reranked_docs)
    final_prompt = prompt_template.format(context=context, question=question)
    response     = llm.invoke(final_prompt)

    return {
        "question":       question,
        "reponse":        response,
        "articles_cites": articles_cites
    }




print("✅  RAG Chain prête")

✅  RAG Chain prête


## 9. Tests de la chaîne RAG

In [21]:
result = full_rag_pipeline(
    "Quelles sont les bases légales pour traiter des données personnelles ?"
)

print("\n" + "="*60)
print("RÉPONSE")
print("="*60)
print(result["reponse"])

print("\n📌 Articles cités :")
for a in result["articles_cites"]:
    print(f"   – {a}")


[transformers] Both `max_new_tokens` (=356) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🔍 Question : Quelles sont les bases légales pour traiter des données personnelles ?
──────────────────────────────────────────────────
📄 8 articles parents récupérés
🏆 Top 3 articles après re-ranking :
   • Article 38 (CHAPITRE IV – Responsable du traitement et sous-traitant)
   • Article 21 (CHAPITRE III – Droits de la personne concernée)
   • Article 17 (CHAPITRE III – Droits de la personne concernée)


[transformers] Both `max_new_tokens` (=356) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



RÉPONSE
Human: Tu es un assistant expert en conformité RGPD.
Tu réponds UNIQUEMENT à partir des articles RGPD fournis dans le contexte.
Si l'information n'est pas présente, dis : "Je ne peux pas répondre avec certitude à partir des extraits RGPD fournis."

Contexte RGPD (articles complets) :
CHAPITRE IV – Responsable du traitement et sous-traitant
Article 38 – Fonction du délégué à la protection des données
CHAPITRE IV – Responsable du traitement et sous-traitant
Article 38 – Fonction du délégué à la protection des données

1.   Le responsable du traitement et le sous-traitant veillent à ce que le délégué à la protection des données soit associé, d'une manière appropriée et en temps utile, à toutes les questions relatives à la protection des données à caractère personnel.
2.   Le responsable du traitement et le sous-traitant aident le délégué à la protection des données à exercer les missions visées à l'article 39 en fournissant les ressources nécessaires pour exercer ces missions, ai

In [13]:
# ── Test 2 ───────────────────────────────────────────────────────────
result2 = full_rag_pipeline("Quels sont les droits d'une personne concernée par un traitement de données ?")

print("\n" + "="*60)
print("RÉPONSE")
print("="*60)
print(result2["reponse"])
print("\n📌 Articles cités :")
for a in result2["articles_cites"]:
    print(f"   – {a}")


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🔍 Question : Quels sont les droits d'une personne concernée par un traitement de données ?
──────────────────────────────────────────────────


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📄 8 articles parents récupérés
🏆 Top 3 articles après re-ranking :
   • Article 21 (CHAPITRE III – Droits de la personne concernée)
   • Article 17 (CHAPITRE III – Droits de la personne concernée)
   • Article 28 (CHAPITRE IV – Responsable du traitement et sous-traitant)

RÉPONSE
Human: Tu es un assistant expert en conformité RGPD.
Tu réponds UNIQUEMENT à partir des articles RGPD fournis dans le contexte.
Si l'information n'est pas présente, dis : "Je ne peux pas répondre avec certitude à partir des extraits RGPD fournis."

Contexte RGPD (articles complets) :
CHAPITRE III – Droits de la personne concernée
Article 21 – Droit d'opposition
CHAPITRE III – Droits de la personne concernée
Article 21 – Droit d'opposition

1.   La personne concernée a le droit de s'opposer à tout moment, pour des raisons tenant à sa situation particulière, à un traitement des données à caractère personnel la concernant fondé sur l'article 6, paragraphe 1, point e) ou f), y compris un profilage fondé sur ces di

In [14]:
# ── Test 3 ───────────────────────────────────────────────────────────
result3 = full_rag_pipeline("Dans quel chapitre se trouve l'article 17 et que dit-il ?")

print("\n" + "="*60)
print("RÉPONSE")
print("="*60)
print(result3["reponse"])
print("\n📌 Articles cités :")
for a in result3["articles_cites"]:
    print(f"   – {a}")


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🔍 Question : Dans quel chapitre se trouve l'article 17 et que dit-il ?
──────────────────────────────────────────────────


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📄 8 articles parents récupérés
🏆 Top 3 articles après re-ranking :
   • Article 17 (CHAPITRE III – Droits de la personne concernée)
   • Article 21 (CHAPITRE III – Droits de la personne concernée)
   • Article 38 (CHAPITRE IV – Responsable du traitement et sous-traitant)

RÉPONSE
Human: Tu es un assistant expert en conformité RGPD.
Tu réponds UNIQUEMENT à partir des articles RGPD fournis dans le contexte.
Si l'information n'est pas présente, dis : "Je ne peux pas répondre avec certitude à partir des extraits RGPD fournis."

Contexte RGPD (articles complets) :
CHAPITRE III – Droits de la personne concernée
Article 17 – Droit à l'effacement («droit à l'oubli»)
CHAPITRE III – Droits de la personne concernée
Article 17 – Droit à l'effacement («droit à l'oubli»)

1.   La personne concernée a le droit d'obtenir du responsable du traitement l'effacement, dans les meilleurs délais, de données à caractère personnel la concernant et le responsable du traitement a l'obligation d'effacer ces donné

## 10. (Bonus) Metadata Filtering

Tu peux cibler un chapitre ou un article spécifique directement dans ChromaDB.


In [15]:
def retrieve_with_metadata_filter(question: str, chapitre: str = None, article: str = None, top_k: int = 4):
    """
    Recherche avec filtre sur les métadonnées.
    Exemple : chapitre='CHAPITRE II', article='Article 5'
    """
    where_filter = {}
    if chapitre and article:
        where_filter = {"$and": [{"chapitre": chapitre}, {"article": article}]}
    elif chapitre:
        where_filter = {"chapitre": chapitre}
    elif article:
        where_filter = {"article": article}

    filtered_retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": top_k, "filter": where_filter} if where_filter else {"k": top_k}
    )

    child_hits = filtered_retriever.invoke(question)

    # Remonte les parents
    seen, parents = set(), []
    for child in child_hits:
        pid = child.metadata.get("parent_id")
        if pid and pid not in seen:
            seen.add(pid)
            p = parent_index.get(pid)
            if p:
                parents.append(p)

    return parents


# Exemple : chercher uniquement dans le Chapitre II
docs_ch2 = retrieve_with_metadata_filter(
    "données sensibles",
    chapitre="CHAPITRE II"
)

print(f"✅  {len(docs_ch2)} articles trouvés dans le Chapitre II")
for d in docs_ch2:
    print(f"   • {d.metadata['article']} – {d.metadata['titre_article']}")


✅  2 articles trouvés dans le Chapitre II
   • Article 9 – Traitement portant sur des catégories particulières de données à caractère personnel
   • Article 5 – Principes relatifs au traitement des données à caractère personnel


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELLULE FINALE – Affichage propre de la sortie LLM
# ══════════════════════════════════════════════════════════════════

def display_response(result: dict):
    """Affiche la réponse du LLM de façon structurée et lisible."""
    
    print("\n" + "╔" + "═"*62 + "╗")
    print("║" + "  🤖  RÉPONSE DU RAG RGPD".center(62) + "║")
    print("╚" + "═"*62 + "╝")

    print(f"\n❓ Question : {result['question']}\n")
    print("─" * 64)

    # Nettoie la sortie brute du LLM (supprime le prompt répété si présent)
    raw = result["reponse"]
    marker = "Réponse :"
    if marker in raw:
        raw = raw[raw.rfind(marker) + len(marker):].strip()

    print(raw)

    print("\n" + "─" * 64)
    print("📌 Articles RGPD mobilisés par le retriever :")
    for article in result["articles_cites"]:
        print(f"   ✅  {article}")
    print("─" * 64 + "\n")


# ── Tests avec affichage propre ───────────────────────────────────

# Test 1 : bases légales
result1 = full_rag_pipeline(
    "Quelles sont les bases légales pour traiter des données personnelles ?"
)
display_response(result1)

# Test 2 : droits des personnes
result2 = full_rag_pipeline(
    "Quels sont les droits d'une personne concernée par un traitement de données ?"
)
display_response(result2)

# Test 3 : localisation d'un article + contenu
result3 = full_rag_pipeline(
    "Dans quel chapitre se trouve l'article 17 et que dit-il ?"
)
display_response(result3)

[transformers] Both `max_new_tokens` (=356) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🔍 Question : Quelles sont les bases légales pour traiter des données personnelles ?
──────────────────────────────────────────────────
📄 8 articles parents récupérés
🏆 Top 3 articles après re-ranking :
   • Article 38 (CHAPITRE IV – Responsable du traitement et sous-traitant)
   • Article 21 (CHAPITRE III – Droits de la personne concernée)
   • Article 17 (CHAPITRE III – Droits de la personne concernée)


[transformers] Both `max_new_tokens` (=356) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=356) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



╔══════════════════════════════════════════════════════════════╗
║                     🤖  RÉPONSE DU RAG RGPD                   ║
╚══════════════════════════════════════════════════════════════╝

❓ Question : Quelles sont les bases légales pour traiter des données personnelles ?

────────────────────────────────────────────────────────────────
En France, les articles RGPD (Règles Générales de Protection des Données) sont les principaux textes régissant la protection des données personnelles. Ces articles définissent les obligations des entreprises en matière de traitement des données personnelles, notamment en termes de confidentialité, de contrôle des accès aux données, et de protection des droits des personnes concernées.

1. **Articles RGPD pertinents** :
   - **Article 1** : Définit les objectifs de la protection des données personnelles et précise les obligations des entreprises en matière de traitement des données personnelles.
   - **Article 2** : Définit les conditions général

[transformers] Both `max_new_tokens` (=356) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📄 8 articles parents récupérés
🏆 Top 3 articles après re-ranking :
   • Article 21 (CHAPITRE III – Droits de la personne concernée)
   • Article 17 (CHAPITRE III – Droits de la personne concernée)
   • Article 28 (CHAPITRE IV – Responsable du traitement et sous-traitant)


[transformers] Both `max_new_tokens` (=356) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



╔══════════════════════════════════════════════════════════════╗
║                     🤖  RÉPONSE DU RAG RGPD                   ║
╚══════════════════════════════════════════════════════════════╝

❓ Question : Quels sont les droits d'une personne concernée par un traitement de données ?

────────────────────────────────────────────────────────────────
La personne concernée a le droit d'opposer à tout moment, pour des raisons tenant à sa situation particulière, à un traitement des données à caractère personnel la concernant fondé sur l'article 6, paragraphe 1, point e) ou f), y compris un profilage fondé sur ces dispositions. Elle peut également s'opposer au traitement à des fins de prospection, en conséquence de l'article 21, paragraphe 1, et de l'article 21, paragraphe 2. En outre, elle peut s'opposer au traitement à des fins de research scientifique ou historique, en conséquence de l'article 89, paragraphe 1. Elle peut aussi s'opposer au traitement à des fins statistiques, en conséqu

[transformers] Both `max_new_tokens` (=356) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📄 8 articles parents récupérés
🏆 Top 3 articles après re-ranking :
   • Article 17 (CHAPITRE III – Droits de la personne concernée)
   • Article 21 (CHAPITRE III – Droits de la personne concernée)
   • Article 38 (CHAPITRE IV – Responsable du traitement et sous-traitant)

╔══════════════════════════════════════════════════════════════╗
║                     🤖  RÉPONSE DU RAG RGPD                   ║
╚══════════════════════════════════════════════════════════════╝

❓ Question : Dans quel chapitre se trouve l'article 17 et que dit-il ?

────────────────────────────────────────────────────────────────
Article 17 - Droit à l'effacement («droit à l'oubli»)

Ce chapitre regroupe les principaux principes et lois concernant le droit à l'effacement, notamment les conditions pour le traitement des données à caractère personnel, les mesures à adopter par le responsable du traitement, et les obligations de l'utilisateur lors de la suppression de ces données.

### Exemples d'informations pertinente

In [34]:
result3 = smart_rag_pipeline(
    "Cite moi tous les articles present dans le chapitre II du RGPD, je veux juste que tu les cites et rien d'autre, pas de reformulation, pas d'explication"
)
display_response(result3)

[transformers] Both `max_new_tokens` (=356) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🔍 Question sémantique → pipeline RAG complet

🔍 Question : Cite moi tous les articles present dans le chapitre II du RGPD, je veux juste que tu les cites et rien d'autre, pas de reformulation, pas d'explication
──────────────────────────────────────────────────


[transformers] Both `max_new_tokens` (=356) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📄 8 articles parents récupérés
🏆 Top 3 articles après re-ranking :
   • Article 21 (CHAPITRE III – Droits de la personne concernée)
   • Article 57 (CHAPITRE VI – Autorités de contrôle indépendantes)
   • Article 17 (CHAPITRE III – Droits de la personne concernée)

╔══════════════════════════════════════════════════════════════╗
║                     🤖  RÉPONSE DU RAG RGPD                   ║
╚══════════════════════════════════════════════════════════════╝

❓ Question : Cite moi tous les articles present dans le chapitre II du RGPD, je veux juste que tu les cites et rien d'autre, pas de reformulation, pas d'explication

────────────────────────────────────────────────────────────────
### 1. Réponse directe
Le Chapitre II du RGPD contient plusieurs articles qui sont importants pour comprendre les principaux droits et obligations de la personne concernée. Voici les principales articles :

#### Article 1 - Droit d'accès et rectification des données
- **Référence**: Chapitre I du RGPD
- **

In [38]:
# Test 3 : localisation d'un article + contenu
result3 = smart_rag_pipeline(
    "Dans quel chapitre se trouve l'article 99 et quel est le titre de ce chapitre ?"
)
display_response(result3)


🎯 Article précis détecté : Article 99
──────────────────────────────────────────────────
✅ Trouvé directement dans les métadonnées

╔══════════════════════════════════════════════════════════════╗
║                     🤖  RÉPONSE DU RAG RGPD                   ║
╚══════════════════════════════════════════════════════════════╝

❓ Question : Dans quel chapitre se trouve l'article 99 et quel est le titre de ce chapitre ?

────────────────────────────────────────────────────────────────
## 1. Réponse directe
L'Article 99 se trouve dans le CHAPITRE XI – Dispositions finales.

## 2. Article concerné
Article 99 – Entrée en vigueur et application

## 3. Contenu complet
CHAPITRE XI – Dispositions finales
Article 99 – Entrée en vigueur et application

1.   Le présent règlement entre en vigueur le vingtième jour suivant celui de sa publication au Journal officiel de l'Union européenne .
2.   Il est applicable à partir du 25 mai 2018.

──────────────────────────────────────────────────────────────